In [1]:
from dotenv import load_dotenv
load_dotenv()
from src.data_loader import load_ground_truth
from src.search import _build_vector_index, _build_patient_id_map
from src.search_dynamic import estimate_n_results, search_dynamic
from src.evaluation import retrieval_recall
import pandas as pd

_build_patient_id_map()
_build_vector_index()

df = load_ground_truth()
test_df = df[df['split'] == 'test'].reset_index(drop=True)
test_df = test_df[test_df['true_fhir_ids'].apply(lambda x: len(x) > 0)]

def flatten_true_ids(true_fhir_ids):
    result = []
    for rtype, ids in true_fhir_ids.items():
        for fid in ids:
            result.append(fid if "/" in fid else f"{rtype}/{fid}")
    return result

results = []
for _, row in test_df.iterrows():
    true_ids = flatten_true_ids(row['true_fhir_ids'])
    n = estimate_n_results(row['question'])
    result = search_dynamic(row['question'])
    recall = retrieval_recall(result['retrieved_ids'], true_ids)
    results.append({
        'question': row['question'][:60],
        'n_estimated': n,
        'n_true_ids': len(true_ids),
        'recall': recall
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values('recall').head(20).to_string())

/Users/rnorel/.pyenv/versions/3.11.14/envs/CH/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rnorel/Documents/Learning/Counsel/research-scientist-interview-main/src/search.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patient ID map built: 100 patients
Vector index loaded: 928935 documents
                                                         question  n_estimated  n_true_ids  recall
299  During this month, when was the first time that patient 1000           50           2     0.0
75   What was the dose of heparin flush (10 units/ml) that was pr           50           2     0.0
74   How much docusate sodium (liquid) dose has been prescribed t         1200           2     0.0
261  When did patient 10023117 for the first time get discharged            50           1     0.0
72   What was the admission type for patient 10019172's first hos           50           1     0.0
71   What was patient 10013049's length of stay in days in the la           50           1     0.0
70   When was the first time when patient 10038933 received a oth           50           1     0.0
69   When was patient 10019172 first discharged from the hospital          300           1     0.0
68   How many unique prescription dr

In [2]:
import json
with open("data/fhir_records/Encounter.ndjson") as f:
    enc = json.loads(f.readline())
print(json.dumps(enc, indent=2))

{
  "id": "9c4ef2ae-dd61-5efe-885d-2a2f0816646f",
  "meta": {
    "profile": [
      "http://mimic.mit.edu/fhir/mimic/StructureDefinition/mimic-encounter"
    ]
  },
  "type": [
    {
      "coding": [
        {
          "code": "308335008",
          "system": "http://snomed.info/sct",
          "display": "Patient encounter procedure"
        }
      ]
    }
  ],
  "class": {
    "code": "AMB",
    "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
    "display": "ambulatory"
  },
  "period": {
    "end": "2180-05-07T17:15:00-04:00",
    "start": "2180-05-06T22:23:00-04:00"
  },
  "status": "finished",
  "subject": {
    "reference": "Patient/0a8eebfd-a352-522e-89f0-1d4a13abdebc"
  },
  "location": [
    {
      "period": {
        "end": "2180-05-06T23:30:00-04:00",
        "start": "2180-05-06T19:17:00-04:00"
      },
      "location": {
        "reference": "Location/501cd59a-cd8a-5f98-8298-2ca9c897d59f"
      }
    },
    {
      "period": {
        "end": "2180-05-07

In [3]:
from src.search import extract_patient_uuid_from_record
import json

with open("data/fhir_records/Encounter.ndjson") as f:
    enc = json.loads(f.readline())

print(extract_patient_uuid_from_record(enc))

0a8eebfd-a352-522e-89f0-1d4a13abdebc


In [5]:
from src.search_patient_filtered import search_patient_filtered
from src.search import _build_patient_id_map

_build_patient_id_map()

q = "When was the first hospital discharge time of patient 10025463?"
result = search_patient_filtered(q, n_results=100)
print("Clinical ID extracted:", result['clinical_id'])
print("Patient UUID:", result['patient_uuid'])
print("Retrieved:", result['retrieved_ids'][:5])
print("Resource types:", set([fid.split('/')[0] for fid in result['retrieved_ids']]))

Patient ID map built: 100 patients
Clinical ID extracted: 10025463
Patient UUID: 28776290-4349-56d3-8c13-adc554feabb8
Retrieved: ['Encounter/ddfdec3b-1cd3-5016-8055-29f764ca34b3', 'Encounter/99cbc731-be46-573e-9b03-344fc87ef579', 'Encounter/7856af3e-7e75-513a-b23f-5dccc85c3bff', 'Procedure/034b7da4-7129-5e73-8904-75aaf442cf61', 'Encounter/8f0c87c8-f6fe-5611-bbdc-61cabf1895f9']
Resource types: {'Observation', 'Encounter', 'Specimen', 'Procedure'}


In [6]:
from src.data_loader import load_ground_truth

df = load_ground_truth()
test_df = df[df['split'] == 'test'].reset_index(drop=True)

# Buscar la pregunta de discharge time para ese paciente
discharge_q = test_df[test_df['question'].str.contains('discharge') & test_df['question'].str.contains('10025463')]
print(discharge_q[['question', 'true_fhir_ids']].to_string())

Empty DataFrame
Columns: [question, true_fhir_ids]
Index: []


In [7]:
discharge_q = test_df[test_df['question'].str.contains('discharge')]
print(discharge_q[['question', 'true_fhir_ids']].head(3).to_string())

                                                                                                              question                                                      true_fhir_ids
94                                                       When was patient 10019172 first discharged from the hospital?  {'Encounter': ['Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2']}
311  How many days have passed since patient 10039831's last stay in careunit discharge lounge in this hospital visit?                                                                 {}
347                                     When did patient 10023117 for the first time get discharged from the hospital?  {'Encounter': ['Encounter/06fc5c8f-ad30-516c-9a74-cdbb491d1407']}


In [8]:
q = "When was patient 10019172 first discharged from the hospital?"
result = search_patient_filtered(q, n_results=100)
print("Retrieved types:", set([fid.split('/')[0] for fid in result['retrieved_ids']]))
print("True ID: Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2")
print("In retrieved:", "Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2" in result['retrieved_ids'])

Retrieved types: {'MedicationAdministration'}
True ID: Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2
In retrieved: False


In [9]:
import json

with open("data/fhir_records/Encounter.ndjson") as f:
    for line in f:
        enc = json.loads(line)
        if "10019172" in json.dumps(enc):
            print(json.dumps(enc, indent=2)[:500])
            break

In [10]:
from src.search import _patient_id_map
uuid = _patient_id_map.get("10019172", "NOT FOUND")
print(uuid)

# Buscar por UUID en Encounter
with open("data/fhir_records/Encounter.ndjson") as f:
    for line in f:
        enc = json.loads(line)
        if uuid in json.dumps(enc):
            print(json.dumps(enc, indent=2)[:300])
            break

76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32
{
  "id": "4238a38e-3f01-58b1-b3f7-306b958412a2",
  "meta": {
    "profile": [
      "http://mimic.mit.edu/fhir/mimic/StructureDefinition/mimic-encounter"
    ]
  },
  "type": [
    {
      "coding": [
        {
          "code": "308335008",
          "system": "http://snomed.info/sct",
          "


In [11]:
from src.search import extract_patient_uuid_from_record
import json

with open("data/fhir_records/Encounter.ndjson") as f:
    for line in f:
        enc = json.loads(line)
        if "4238a38e-3f01-58b1-b3f7-306b958412a2" in line:
            uuid = extract_patient_uuid_from_record(enc)
            print("Extracted UUID:", uuid)
            print("Expected UUID:", "76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32")
            break

Extracted UUID: 76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32
Expected UUID: 76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32


In [12]:
import src.search as _search_module
from src.search import _build_vector_index
_build_vector_index()

results = _search_module._chroma_collection.get(
    where={"fhir_id": {"$eq": "Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2"}},
    include=["metadatas"]
)
print(results['metadatas'])

Vector index loaded: 928935 documents
[{'fhir_id': 'Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2', 'patient_uuid': '76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32'}]


In [13]:
uuid = "76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32"
results = _search_module._chroma_collection.get(
    where={"patient_uuid": {"$eq": uuid}},
    include=["metadatas"],
    limit=5
)
print(f"Found: {len(results['ids'])} documents")
print(results['metadatas'][:3])

Found: 5 documents
[{'patient_uuid': '76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32', 'fhir_id': 'Condition/4d760636-6773-5ac1-8da4-05a7b4f01710'}, {'patient_uuid': '76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32', 'fhir_id': 'Condition/533eaad5-a8ca-5754-9335-4974f6ec5289'}, {'fhir_id': 'Condition/b08f0cea-abea-5031-b1e7-6e519f232312', 'patient_uuid': '76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32'}]


In [14]:
results = _search_module._chroma_collection.get(
    where={"patient_uuid": {"$eq": ""}},
    include=["metadatas"],
    limit=5
)
print(f"Documents with empty patient_uuid: {len(results['ids'])}")
print(results['metadatas'][:3])

Documents with empty patient_uuid: 5
[{'patient_uuid': '', 'fhir_id': 'Location/ecbf468a-22ec-5320-8e11-6ebcc918dad5'}, {'fhir_id': 'Location/501cd59a-cd8a-5f98-8298-2ca9c897d59f', 'patient_uuid': ''}, {'fhir_id': 'Location/f4ed77ed-3be3-5700-bc6a-3622ca10f90f', 'patient_uuid': ''}]


In [15]:
results = _search_module._chroma_collection.get(
    where={"patient_uuid": {"$eq": "76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32"}},
    include=["metadatas"],
    limit=100
)
print(f"Found: {len(results['ids'])} documents")

Found: 100 documents


In [16]:
import openai
import os

oai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
q = "When was patient 10019172 first discharged from the hospital?"
embedding = oai_client.embeddings.create(
    input=[q],
    model="text-embedding-3-small"
).data[0].embedding

results = _search_module._chroma_collection.query(
    query_embeddings=[embedding],
    n_results=100,
    where={"patient_uuid": {"$eq": "76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32"}}
)

retrieved = [meta["fhir_id"] for meta in results["metadatas"][0]]
print("Resource types:", set([fid.split('/')[0] for fid in retrieved]))
print("True ID in results:", "Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2" in retrieved)

Resource types: {'Condition', 'Procedure', 'Observation', 'Encounter', 'Specimen', 'Patient'}
True ID in results: True


In [17]:
from src.search_patient_filtered import search_patient_filtered, extract_clinical_patient_id
from src.search import _patient_id_map

q = "When was patient 10019172 first discharged from the hospital?"
clinical_id = extract_clinical_patient_id(q)
patient_uuid = _patient_id_map.get(clinical_id, "")
print("Clinical ID:", clinical_id)
print("Patient UUID:", patient_uuid)

result = search_patient_filtered(q, n_results=100)
print("UUID used in search:", result['patient_uuid'])
print("True ID found:", "Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2" in result['retrieved_ids'])

Clinical ID: 10019172
Patient UUID: 76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32
UUID used in search: 76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32
True ID found: False


In [18]:
from src.search import expand_query
q = "When was patient 10019172 first discharged from the hospital?"
expanded = expand_query(q)
print("Expanded query:", expanded)

Expanded query: When was patient 76202c51-1b9d-5cc2-a7bc-3dfb2ac3ab32 first discharged from the hospital?


In [19]:
result = search_patient_filtered(q, n_results=500)
print("True ID found:", "Encounter/4238a38e-3f01-58b1-b3f7-306b958412a2" in result['retrieved_ids'])
print("Resource types:", set([fid.split('/')[0] for fid in result['retrieved_ids']]))

True ID found: True
Resource types: {'Condition', 'Procedure', 'Observation', 'Encounter', 'Specimen', 'MedicationDispense', 'MedicationRequest', 'MedicationAdministration'}


In [20]:
import json

with open("data/fhir_records/ObservationOutputevents.ndjson") as f:
    obs = json.loads(f.readline())
print(json.dumps(obs, indent=2))

{
  "id": "eb1e2eab-10f6-5d76-b489-24c8994e65c6",
  "code": {
    "coding": [
      {
        "code": "226560",
        "system": "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-d-items",
        "display": "Void"
      }
    ]
  },
  "meta": {
    "profile": [
      "http://mimic.mit.edu/fhir/mimic/StructureDefinition/mimic-observation-outputevents"
    ]
  },
  "issued": "2180-07-23T16:00:00-04:00",
  "status": "final",
  "subject": {
    "reference": "Patient/0a8eebfd-a352-522e-89f0-1d4a13abdebc"
  },
  "category": [
    {
      "coding": [
        {
          "code": "Output",
          "system": "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-observation-category"
        }
      ]
    }
  ],
  "encounter": {
    "reference": "Encounter/18f74cab-e07d-5ca6-98b8-01f8f9956caa"
  },
  "resourceType": "Observation",
  "valueQuantity": {
    "code": "ml",
    "unit": "ml",
    "value": 175,
    "system": "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-units"
  },
  "effectiveDateT

In [21]:
import json

# Los peores templates eran: discharge, careunit, drug dose, output
# Veamos ObservationMicroTest también

for filename in ["ObservationMicroOrg.ndjson", "ObservationMicroTest.ndjson", "SpecimenLab.ndjson"]:
    print(f"\n=== {filename} ===")
    with open(f"data/fhir_records/{filename}") as f:
        rec = json.loads(f.readline())
    print(list(rec.keys()))


=== ObservationMicroOrg.ndjson ===
['id', 'code', 'meta', 'status', 'subject', 'category', 'hasMember', 'derivedFrom', 'resourceType', 'effectiveDateTime']

=== ObservationMicroTest.ndjson ===
['id', 'code', 'meta', 'status', 'subject', 'category', 'specimen', 'valueString', 'resourceType', 'effectiveDateTime']

=== SpecimenLab.ndjson ===
['id', 'meta', 'type', 'subject', 'collection', 'identifier', 'resourceType']


In [22]:
print("=== ObservationMicroTest valueString ===")
with open("data/fhir_records/ObservationMicroTest.ndjson") as f:
    for line in f:
        rec = json.loads(line)
        vs = rec.get("valueString", "")
        if vs:
            print(vs)
            break

print("\n=== SpecimenLab type ===")
with open("data/fhir_records/SpecimenLab.ndjson") as f:
    rec = json.loads(f.readline())
    print(json.dumps(rec.get("type", {}), indent=2))

=== ObservationMicroTest valueString ===
POSITIVE BY EIA.  

=== SpecimenLab type ===
{
  "coding": [
    {
      "code": "Blood",
      "system": "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-lab-fluid"
    }
  ]
}


In [23]:
import json
import glob

# Ver todos los resource types y sus campos únicos
for filepath in sorted(glob.glob("data/fhir_records/*.ndjson")):
    with open(filepath) as f:
        rec = json.loads(f.readline())
    rtype = rec.get("resourceType", "")
    fields = list(rec.keys())
    print(f"{rtype}: {fields}")

Condition: ['id', 'code', 'meta', 'subject', 'category', 'encounter', 'resourceType']
Condition: ['id', 'code', 'meta', 'subject', 'category', 'encounter', 'resourceType']
Encounter: ['id', 'meta', 'type', 'class', 'period', 'status', 'subject', 'location', 'priority', 'identifier', 'serviceType', 'resourceType', 'hospitalization', 'serviceProvider']
Encounter: ['id', 'meta', 'type', 'class', 'partOf', 'period', 'status', 'subject', 'identifier', 'resourceType', 'hospitalization', 'serviceProvider']
Encounter: ['id', 'meta', 'type', 'class', 'partOf', 'period', 'status', 'subject', 'location', 'identifier', 'resourceType']
Location: ['id', 'meta', 'name', 'status', 'physicalType', 'resourceType', 'managingOrganization']
Medication: ['id', 'code', 'meta', 'status', 'identifier', 'resourceType']
MedicationAdministration: ['id', 'meta', 'dosage', 'status', 'context', 'request', 'subject', 'resourceType', 'effectiveDateTime', 'medicationCodeableConcept']
MedicationAdministration: ['id', 'm